# Experiment 4: Pause while writing, change the model's memory, resume

When GPT-2 writes text one word at a time, it does not have to reread the whole sentence for every new word.
In each of its 12 blocks it keeps a **memory** of the words seen so far (called the **KV cache**). For every word it stores:
- a **key**: a label saying what this word offers, which later words compare against to decide where to look, and
- a **value**: the information this word hands over to any later word that looks at it.

A new word only needs its own pass through the model. Everything about the earlier words is read from the memory.
So the memory **is** the paused state of the model: save it, and you can resume later by feeding in only the next word.

In this notebook we:
1. Check that writing with memory gives exactly the same answer as writing without it.
2. **PAUSE** before the last word, and **save** the memory of all earlier words.
3. **RESUME** by feeding in only the last word, and prove that earlier words are **not rerun**.
4. **MODIFY** the saved memory for ' Paris': replace it with the ' Berlin' one, or switch it off.
5. Find which blocks and which half of the memory (keys or values) matter.
6. Let the model keep writing from a modified memory.

## Step 0: Install (Colab only)
Tested with PyTorch 2.14.1 and Transformers 5.20.2. Uncomment only if something breaks.

In [ ]:
# !pip install -q transformers==5.20.2

## Step 1: Load GPT-2 small

In [1]:
import copy
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_grad_enabled(False)  # prediction only, no training
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
gpt2 = model.transformer
NUM_BLOCKS = len(gpt2.h)

print("Device:", device)
print("Number of blocks:", NUM_BLOCKS)
print("Look-around heads per block:", model.config.n_head)

Device: cpu
Number of blocks: 12
Look-around heads per block: 12


## Step 2: Normal run (no memory)
This is the reference answer. Everything later is compared against it.

In [2]:
PROMPT = "I live in Paris, so I speak"
OTHER_PROMPT = "I live in Berlin, so I speak"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
other_ids = tokenizer(OTHER_PROMPT, return_tensors="pt").input_ids.to(device)
assert other_ids.shape == input_ids.shape
words = [tokenizer.decode(t) for t in input_ids[0]]
print("Word-pieces:", words)

PARIS_POS, LAST_POS = 3, len(words) - 1
FRENCH, GERMAN = tokenizer.encode(" French")[0], tokenizer.encode(" German")[0]

def top_predictions(logits, k=5):
    probs = logits[0, -1].softmax(-1)
    values, indices = probs.topk(k)
    return [(tokenizer.decode(i), v.item()) for i, v in zip(indices, values)]

def chances(logits):
    p = logits[0, -1].softmax(-1)
    return tokenizer.decode(p.argmax()), p[FRENCH].item(), p[GERMAN].item()

def compare(new_logits, title="Modified model"):
    """Print the normal and new top 5 side by side, plus the chance of ' French' and ' German'."""
    print(f"{'Normal model':<26}{title}")
    for (w1, p1), (w2, p2) in zip(top_predictions(normal_logits), top_predictions(new_logits)):
        print(f"  {w1!r:12} {p1:6.2%}      {w2!r:12} {p2:6.2%}")
    _, f0, g0 = chances(normal_logits)
    _, f1, g1 = chances(new_logits)
    print(f"\n  ' French': {f0:6.2%} -> {f1:6.2%}")
    print(f"  ' German': {g0:6.2%} -> {g1:6.2%}")

normal_logits = model(input_ids, use_cache=False).logits
print("\nNormal model - top 5 next words:")
for word, p in top_predictions(normal_logits):
    print(f"  {word!r:12} {p:6.2%}")

Word-pieces: ['I', ' live', ' in', ' Paris', ',', ' so', ' I', ' speak']

Normal model - top 5 next words:
  ' French'    63.05%
  ' English'    4.93%
  ' to'         3.16%
  ' a'          2.94%
  ' with'       1.76%


## Step 3: Pausing and resuming with memory
- `run_until(ids, n)` runs the first `n` word-pieces through the model, then **PAUSES**.
  It returns a **checkpoint**: the memory of those words (one key table and one value table per block).
- `resume(checkpoint, ids)` **RESUMES** by feeding in only the new word-pieces `ids`. Earlier words are read from the memory, never rerun.
- `copy_ckpt(checkpoint)` makes a separate copy. This matters because resuming **adds the new word to the memory**,
  so we always resume from a copy and keep the saved checkpoint untouched.
- `keys(ckpt, i)` and `values(ckpt, i)` give the memory of block `i`: shape (1 sentence, 12 heads, words, 64 numbers).

In [3]:
def run_until(input_ids, n):
    """Run the first n word-pieces, then PAUSE. Returns the memory (KV cache) as the checkpoint."""
    out = model(input_ids[:, :n], use_cache=True)
    return out.past_key_values

def resume(checkpoint, new_ids):
    """RESUME: feed only the new word-pieces; earlier words come from the memory. Returns (logits, memory)."""
    out = model(new_ids, past_key_values=checkpoint, use_cache=True)
    return out.logits, out.past_key_values

def copy_ckpt(checkpoint):
    """Separate copy of a checkpoint, so resuming (which adds to the memory) leaves the saved one untouched."""
    return copy.deepcopy(checkpoint)

def keys(checkpoint, i):
    return checkpoint.layers[i].keys

def values(checkpoint, i):
    return checkpoint.layers[i].values

## Step 4: A counter that records how many word-pieces each block processes
`block 5: 1` means block 5 ran on 1 word-piece. A full run of the 8-word sentence shows `8` for every block.
We use this to **prove** that resuming only processes the new word.

In [4]:
for h in globals().get("counter_hooks", []):  # safe to rerun this cell
    h.remove()

words_processed = []
counter_hooks = [
    block.ln_1.register_forward_hook(lambda m, inp, out, i=i: words_processed.append((i, out.shape[1])))
    for i, block in enumerate(gpt2.h)
]
print("Counter attached to", len(counter_hooks), "blocks")

def only_new_words(n):
    """True if every block ran exactly once, on exactly n word-pieces."""
    return words_processed == [(i, n) for i in range(NUM_BLOCKS)]

Counter attached to 12 blocks


## Step 5: Check that running with memory matches the normal model
We feed the sentence in one word at a time, each time reading the earlier words from memory.
The final prediction must match the normal run, which read the whole sentence at once.

Tiny differences (around 0.0002, on word scores of about 100) are expected: the same sums are added up in a different order,
which changes the last few digits. So from here on we allow differences up to 0.001.

In [5]:
memory = None
for pos in range(input_ids.shape[1]):
    words_processed.clear()
    logits, memory = resume(memory, input_ids[:, pos:pos + 1])
    assert only_new_words(1), "A block processed more than the new word"

diff = (logits[0, -1] - normal_logits[0, -1]).abs().max().item()
print("Fed in", input_ids.shape[1], "word-pieces one at a time; every block processed only 1 word-piece per step")
print("Largest difference from normal model:", diff)
assert diff < 1e-3, "Running with memory does not match the normal model"
print("OK: running with memory matches the normal model")

Fed in 8 word-pieces one at a time; every block processed only 1 word-piece per step
Largest difference from normal model: 0.000213623046875
OK: running with memory matches the normal model


## Step 6: PAUSE before the last word and SAVE the memory
We run *"I live in Paris, so I"* (everything except ' speak') and pause.
The checkpoint is the memory of those 7 word-pieces in all 12 blocks.

In [6]:
PAUSE_AT = LAST_POS  # number of word-pieces run before pausing (all except ' speak')
last_word = input_ids[:, LAST_POS:]

words_processed.clear()
ckpt = run_until(input_ids, PAUSE_AT)  # <-- PAUSE happens here
saved = copy_ckpt(ckpt)                # <-- SAVE a separate copy

print("PAUSED after:", words[:PAUSE_AT])
print("Word-pieces processed per block:", words_processed[0][1])
print("Memory of one block: keys", tuple(keys(saved, 0).shape), "+ values", tuple(values(saved, 0).shape))
print("  (1 sentence, 12 heads, 7 word-pieces, 64 numbers each)")
total = sum(keys(saved, i).numel() + values(saved, i).numel() for i in range(NUM_BLOCKS))
print(f"Whole memory: {NUM_BLOCKS} blocks x 2 tables -> {total:,} numbers")

other_saved = run_until(other_ids, PAUSE_AT)  # same pause point for the Berlin sentence
same = [torch.allclose(keys(saved, 0)[0, :, p], keys(other_saved, 0)[0, :, p]) for p in range(PAUSE_AT)]
print("\nBlock 0 memory identical to the Berlin sentence, per word:")
print("  ", {w: s for w, s in zip(words[:PAUSE_AT], same)})

PAUSED after: ['I', ' live', ' in', ' Paris', ',', ' so', ' I']
Word-pieces processed per block: 7
Memory of one block: keys (1, 12, 7, 64) + values (1, 12, 7, 64)
  (1 sentence, 12 heads, 7 word-pieces, 64 numbers each)
Whole memory: 12 blocks x 2 tables -> 129,024 numbers

Block 0 memory identical to the Berlin sentence, per word:
   {'I': True, ' live': True, ' in': True, ' Paris': False, ',': True, ' so': True, ' I': True}


The memory for *"I live in"* is identical in both sentences, because those words only see the words before them.
From ' Paris'/' Berlin' onward the memory differs.

## Step 7: RESUME with the saved memory unchanged (check)
We feed in only ' speak'. Every block should process exactly 1 word-piece, and the answer should match the normal one.

In [7]:
words_processed.clear()
resumed, _ = resume(copy_ckpt(saved), last_word)  # <-- RESUME

print("RESUMED with:", [tokenizer.decode(t) for t in last_word[0]])
print("Word-pieces processed per block:", sorted({n for _, n in words_processed}))
print("Largest difference from normal model:", (resumed[0, -1] - normal_logits[0, -1]).abs().max().item())
assert only_new_words(1), "Something earlier was rerun!"
assert torch.allclose(resumed[0, -1], normal_logits[0, -1], atol=1e-3)
print("OK: resuming gives the normal answer, and the first 7 words were not rerun")

RESUMED with: [' speak']
Word-pieces processed per block: [1]
Largest difference from normal model: 0.0001983642578125
OK: resuming gives the normal answer, and the first 7 words were not rerun


## Step 8: Tools to change the memory
- `swap_in(ckpt, source, positions, ...)` copies the memory of some words from another checkpoint.
- `switch_off(ckpt, positions, ...)` sets the **values** of some words to zero, so they hand over nothing.
  (Their keys stay, so other words still look at them, but receive nothing.)

Both can be limited to some blocks (`blocks=`) and to keys only, values only, or both (`parts=`).
Both return a new checkpoint and leave the input untouched.

In [8]:
ALL_BLOCKS = range(NUM_BLOCKS)

def swap_in(checkpoint, source, positions, blocks=ALL_BLOCKS, parts=("keys", "values")):
    """Copy of `checkpoint` with the memory of `positions` taken from `source`."""
    new = copy_ckpt(checkpoint)
    for i in blocks:
        for part in parts:
            getattr(new.layers[i], part)[0, :, positions] = getattr(source.layers[i], part)[0, :, positions]
    return new

def switch_off(checkpoint, positions, blocks=ALL_BLOCKS):
    """Copy of `checkpoint` with the values of `positions` set to zero."""
    new = copy_ckpt(checkpoint)
    for i in blocks:
        new.layers[i].values[0, :, positions] = 0
    return new

def resume_chances(checkpoint):
    words_processed.clear()
    logits, _ = resume(checkpoint, last_word)
    assert only_new_words(1)
    return chances(logits)

## Step 9: MODIFY — replace the memory of ' Paris' with the ' Berlin' one
Only the memory of the word ' Paris' changes, in all 12 blocks. The memory of ', so I' is left as it is.

In [9]:
new_ckpt = swap_in(saved, other_saved, [PARIS_POS])  # <-- MODIFY

words_processed.clear()
logits_swap_paris, _ = resume(new_ckpt, last_word)  # <-- RESUME
assert only_new_words(1)
compare(logits_swap_paris, "Berlin memory for ' Paris'")

Normal model              Berlin memory for ' Paris'
  ' French'    63.05%      ' German'    60.08%
  ' English'    4.93%      ' English'    5.52%
  ' to'         3.16%      ' a'          4.43%
  ' a'          2.94%      ' to'         3.51%
  ' with'       1.76%      ' the'        2.17%

  ' French': 63.05% ->  1.23%
  ' German':  0.62% -> 60.08%


## Step 10: MODIFY — also replace the memory of the words after ' Paris'
The words ', so I' came **after** ' Paris', so while they were being run they looked at ' Paris' and picked up some of it.
That information now sits in their memory too. Here we swap the memory of ' Paris' **and** every word after it.
Since *"I live in"* is identical in both sentences, this should give exactly the Berlin sentence's answer.

In [10]:
after = list(range(PARIS_POS, PAUSE_AT))
new_ckpt = swap_in(saved, other_saved, after)  # <-- MODIFY

words_processed.clear()
logits_swap_after, _ = resume(new_ckpt, last_word)  # <-- RESUME
assert only_new_words(1)
compare(logits_swap_after, "Berlin memory for ' Paris, so I'")

berlin_logits = model(other_ids, use_cache=False).logits
print("\nLargest difference from the normal Berlin sentence:",
      (logits_swap_after[0, -1] - berlin_logits[0, -1]).abs().max().item())
assert torch.allclose(logits_swap_after[0, -1], berlin_logits[0, -1], atol=1e-3)
print("OK: identical to running the Berlin sentence")

Normal model              Berlin memory for ' Paris, so I'
  ' French'    63.05%      ' German'    63.57%
  ' English'    4.93%      ' English'    4.79%
  ' to'         3.16%      ' a'          4.40%
  ' a'          2.94%      ' to'         3.36%
  ' with'       1.76%      ' the'        1.96%

  ' French': 63.05% ->  0.84%
  ' German':  0.62% -> 63.57%

Largest difference from the normal Berlin sentence: 0.0002288818359375
OK: identical to running the Berlin sentence


## Step 11: MODIFY — switch off the memory of ' Paris'
We set the values of ' Paris' to zero in all blocks, so ' speak' receives nothing from it.

In [11]:
new_ckpt = switch_off(saved, [PARIS_POS])  # <-- MODIFY

words_processed.clear()
logits_off, _ = resume(new_ckpt, last_word)  # <-- RESUME
assert only_new_words(1)
compare(logits_off, "' Paris' switched off")

Normal model              ' Paris' switched off
  ' French'    63.05%      ' English'   15.46%
  ' English'    4.93%      ' Spanish'   12.24%
  ' to'         3.16%      ' to'         7.43%
  ' a'          2.94%      ' French'     5.60%
  ' with'       1.76%      ' for'        3.93%

  ' French': 63.05% ->  5.60%
  ' German':  0.62% ->  2.85%


## Step 12: Which blocks, and which half of the memory, matter?
For the word ' Paris', we swap in the Berlin memory **one block at a time**: keys only, values only, or both.
Then we do the same for ' Paris' and all words after it. Each line is a separate resume from the same saved checkpoint.

In [12]:
print(f"{'':<6}{'--- memory of Paris only ---':^42}{'--- Paris and words after ---':^42}")
print(f"{'Block':<6}" + f"{'keys':>14}{'values':>14}{'both':>14}" * 2)
print(f"{'':<6}" + f"{'French/German':>14}" * 6)
for i in range(NUM_BLOCKS):
    line = f"{i:<6}"
    for positions in ([PARIS_POS], after):
        for parts in (("keys",), ("values",), ("keys", "values")):
            _, f, g = resume_chances(swap_in(saved, other_saved, positions, blocks=[i], parts=parts))
            line += f"{f:>8.0%}/{g:<5.0%}"
    print(line)
_, f, g = chances(normal_logits)
print(f"\nNormal model: French {f:.0%} / German {g:.0%}")

             --- memory of Paris only ---             --- Paris and words after ---       
Block           keys        values          both          keys        values          both
       French/German French/German French/German French/German French/German French/German
0          63%/1%        64%/1%        63%/1%        63%/1%        64%/1%        63%/1%   
1          63%/1%        64%/1%        64%/1%        63%/1%        64%/1%        64%/1%   
2          63%/1%        64%/1%        65%/1%        63%/1%        64%/1%        65%/1%   
3          63%/1%        63%/1%        63%/1%        63%/1%        63%/1%        63%/1%   
4          63%/1%        63%/1%        63%/1%        63%/1%        63%/1%        64%/1%   
5          63%/1%        63%/1%        63%/1%        63%/1%        63%/1%        63%/1%   
6          63%/1%        63%/1%        63%/1%        63%/1%        63%/1%        63%/1%   
7          63%/1%        63%/1%        63%/1%        63%/1%        64%/1%        63%/1%   

## Step 13: Keep writing from a modified memory
Both models write 8 more words. Each new word is one resume step: only that word is processed, and its memory is added.
The modified model starts from the saved checkpoint with the memory of ' Paris, so I' swapped for the Berlin one.
The text it was given still says *"Paris"*; only its memory was changed.

In [13]:
NEW_WORDS = 8

def write_on(checkpoint, first_ids, n=NEW_WORDS):
    """Resume from `checkpoint` with `first_ids`, then keep picking the most likely next word."""
    memory, next_ids, written = copy_ckpt(checkpoint), first_ids, []
    for _ in range(n):
        words_processed.clear()
        logits, memory = resume(memory, next_ids)  # <-- RESUME (one word)
        assert only_new_words(next_ids.shape[1])
        next_ids = logits[:, -1:].argmax(-1)
        written.append(next_ids.item())
    return tokenizer.decode(written)

normal_text = write_on(saved, last_word)
modified_text = write_on(swap_in(saved, other_saved, after), last_word)

# reference: the normal model writing without memory, rereading the whole text for every word
ids = input_ids
for _ in range(NEW_WORDS):
    ids = torch.cat([ids, model(ids, use_cache=False).logits[:, -1:].argmax(-1)], dim=1)
assert normal_text == tokenizer.decode(ids[0, input_ids.shape[1]:])

print("Normal model:  ", repr(PROMPT + normal_text))
print("Modified model:", repr(PROMPT + modified_text))
print("\nOK: writing with memory matches writing without it, and each step processed only one new word")

Normal model:   "I live in Paris, so I speak French. I'm not a French person"
Modified model: "I live in Paris, so I speak German. I'm not a German speaker"

OK: writing with memory matches writing without it, and each step processed only one new word
